# Modul 2 — Models, ORM & Forms

**Lanjutan dari Modul 1** (studi kasus yang sama: aplikasi To-Do List).

## Tujuan Pembelajaran
1. Menjelaskan apa itu ORM dan kenapa dipakai.
2. Mendefinisikan model Django dan menjalankan migration.
3. Melakukan operasi CRUD lewat Django shell (`python manage.py shell`).
4. Mendaftarkan model ke Django Admin.
5. Membuat `ModelForm` untuk input data dari user, dihubungkan ke view + template.

> **Prasyarat:** kamu sudah menyelesaikan Modul 1, jadi folder `proyek-todo-app/` dengan project `todo_project` dan app `tasks` sudah ada. Notebook ini adalah **panduan referensi** — jalankan command di terminal kamu sendiri, salin isi file ke editor, bukan dengan menjalankan cell notebook ini.


## 1. Apa itu ORM?

**ORM (Object-Relational Mapping)** menerjemahkan objek Python jadi baris tabel database, dan sebaliknya — tanpa kamu menulis SQL manual.

Keuntungan ORM Django: kode query jadi Python biasa (`Task.objects.filter(...)`), otomatis kompatibel dengan banyak database (SQLite, PostgreSQL, MySQL), dan setiap perubahan struktur tabel tercatat lewat **migration**.

Setiap **Model** = satu class Python yang merepresentasikan satu tabel. Setiap atribut class = satu kolom.


## 2. Mendefinisikan Model `Task`

Buka `tasks/models.py` dan isi seperti ini:

```python
from django.db import models


class Task(models.Model):
    title = models.CharField(max_length=200)
    description = models.TextField(blank=True)
    is_done = models.BooleanField(default=False)
    created_at = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return self.title
```


## 3. Migrations

```bash
python manage.py makemigrations tasks
python manage.py migrate
```

- `makemigrations tasks` — membaca perubahan di `models.py`, membuat file migration (belum mengubah database).
- `migrate` — menerapkan migration ke database.


## 4. Django Admin

App `tasks` sekarang punya model. Daftarkan ke **`tasks/admin.py`**:

```python
from django.contrib import admin
from .models import Task

admin.site.register(Task)
```

Lalu buat akun admin:

```bash
python manage.py createsuperuser
```

Command ini interaktif — akan menanyakan `Username`, `Email`, dan `Password` satu per satu di terminal.

Jalankan `python manage.py runserver`, buka `http://127.0.0.1:8000/admin/`, dan login pakai akun yang baru dibuat untuk melihat panel adminnya.


## 5. ORM CRUD Dasar Lewat Django Shell

```bash
python manage.py shell
```

Command ini membuka Python interaktif yang sudah otomatis ter-koneksi ke project Django kamu. Di situ kamu bisa coba langsung, satu baris demi satu baris:

```python
>>> from tasks.models import Task

>>> # CREATE
>>> Task.objects.create(title="Belajar Model Django", is_done=True)
>>> Task.objects.create(title="Belajar Migrations", is_done=True)
>>> Task.objects.create(title="Belajar ORM CRUD", is_done=False)

>>> # READ
>>> Task.objects.all()                      # semua data
>>> Task.objects.filter(is_done=False)      # dengan kondisi

>>> # UPDATE
>>> t = Task.objects.get(title="Belajar ORM CRUD")
>>> t.is_done = True
>>> t.save()

>>> # DELETE
>>> t.delete()
```

Ketik `exit()` untuk keluar dari shell.


## 6. Menghubungkan Model ke Views

Ganti **`tasks/views.py`** supaya mengambil data dari database, bukan dict hardcode. Tambahkan juga view untuk menambah tugas (`add_task`) dan menandai selesai/belum (`toggle_task`):

```python
from django.shortcuts import render, redirect, get_object_or_404
from .models import Task
from .forms import TaskForm


def index(request):
    tasks = Task.objects.all().order_by("-created_at")
    context = {"tasks": tasks, "total": tasks.count()}
    return render(request, "tasks/index.html", context)


def add_task(request):
    if request.method == "POST":
        form = TaskForm(request.POST)
        if form.is_valid():
            form.save()
            return redirect("index")
    else:
        form = TaskForm()
    return render(request, "tasks/add_task.html", {"form": form})


def toggle_task(request, task_id):
    task = get_object_or_404(Task, id=task_id)
    task.is_done = not task.is_done
    task.save()
    return redirect("index")
```


## 7. Forms — `ModelForm`

`ModelForm` membuat form HTML otomatis dari sebuah model, lengkap dengan validasi dasar (`title` wajib diisi karena bukan `blank=True`). Buat **`tasks/forms.py`**:

```python
from django import forms
from .models import Task


class TaskForm(forms.ModelForm):
    class Meta:
        model = Task
        fields = ["title", "description"]
```


## 8. Update `tasks/urls.py` untuk Route Baru

```python
from django.urls import path
from . import views

urlpatterns = [
    path('', views.index, name='index'),
    path('add/', views.add_task, name='add_task'),
    path('toggle/<int:task_id>/', views.toggle_task, name='toggle_task'),
]
```


## 9. Update Templates

**`tasks/templates/tasks/index.html`**:

```html
<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <title>To-Do List</title>
</head>
<body>
    <h1>Daftar Tugas Saya</h1>
    <p>Total tugas: {{ total }}</p>
    <a href="{% url 'add_task' %}">+ Tambah tugas</a>
    <ul>
        {% for task in tasks %}
            <li>
                {% if task.is_done %}<s>{{ task.title }}</s>{% else %}{{ task.title }}{% endif %}
                - <a href="{% url 'toggle_task' task.id %}">toggle</a>
            </li>
        {% empty %}
            <li>Belum ada tugas.</li>
        {% endfor %}
    </ul>
</body>
</html>
```

**`tasks/templates/tasks/add_task.html`** (file baru):

```html
<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <title>Tambah Tugas</title>
</head>
<body>
    <h1>Tambah Tugas Baru</h1>
    <form method="post">
        {% csrf_token %}
        {{ form.as_p }}
        <button type="submit">Simpan</button>
    </form>
    <a href="{% url 'index' %}">Kembali</a>
</body>
</html>
```


## 10. Menjalankan Server & Menguji di Browser

```bash
python manage.py runserver
```

Buka `http://127.0.0.1:8000/` di browser. Coba:
1. Klik **"+ Tambah tugas"**, isi form, submit — tugas baru harus muncul di daftar.
2. Klik **"toggle"** di sebelah salah satu tugas — statusnya harus berubah selesai/belum.
3. Buka `http://127.0.0.1:8000/admin/` (login pakai akun superuser tadi) — data yang kamu tambah lewat form tadi harus terlihat juga di panel admin.


## 11. Ringkasan & Keterkaitan dengan Modul 1

- **Modul 1** membangun jalur `urls → views → templates` dengan data statis.
- **Modul 2** mengganti data statis itu dengan **Model** (`Task`) yang tersimpan di database lewat **ORM**.
- Command inti baru: `makemigrations`, `migrate`, `createsuperuser`, `python manage.py shell`.
- **Django Admin** memberi UI siap pakai untuk mengelola data model.
- **`ModelForm`** menghasilkan form HTML + validasi otomatis dari model, dipakai di view `add_task`.

Aplikasi To-Do List sekarang punya alur CRUD dasar: **Create** (form tambah), **Read** (index), **Update** (toggle selesai) — **Delete** lewat UI belum ada, jadi bahan latihan berikut.


## 12. Exercise (dikerjakan saat sesi kelas)

Tambahkan field **`priority`** ke model `Task` dengan pilihan `low`, `medium`, `high` (`choices` pada `CharField`).

1. Tambahkan field `priority` di `tasks/models.py` (`choices`, default `"medium"`).
2. Jalankan lagi `python manage.py makemigrations tasks` dan `python manage.py migrate`.
3. Tambahkan `"priority"` ke `fields` pada `TaskForm` (`tasks/forms.py`).
4. Tampilkan `priority` di `index.html`, misal `[{{ task.priority }}]` di samping judul tugas.
5. Jalankan `runserver`, buka browser, tambah satu tugas dengan priority `high`, pastikan tampil di daftar.


## 13. Homework (dikerjakan di luar sesi kelas)

Lengkapi fitur CRUD To-Do List ini menjadi benar-benar lengkap:

1. Buat view **`delete_task`** (`/delete/<int:task_id>/`) yang menghapus task, lalu redirect ke `index`. Tambahkan link "hapus" di `index.html`.
2. Buat view **`task_detail`** (`/task/<int:task_id>/`) yang menampilkan detail satu task (title, description lengkap, status, tanggal dibuat). Tambahkan link dari `index.html`.
3. Tambahkan validasi: kalau `title` dikosongkan saat submit form tambah tugas, form harus menolak dan menampilkan pesan error (`ModelForm` sudah otomatis melakukan ini — pastikan error-nya ditampilkan di template dengan `{{ form.title.errors }}`).

**Kriteria selesai:** semua fitur di atas bisa dicoba lewat `python manage.py runserver` + browser tanpa error, dan form tambah tugas menolak submit kalau `title` kosong.
